# Build a Bitfinex R0 adapter

Define the protocol in Python, connect a source, and serve its books in the terminal. All of the protocol definition is below.

Start with the short recorded-input example, then connect the real source and open the terminal. Price and quantity outputs use integer units at the declared precisions.

In [1]:
from pathlib import Path
from time import perf_counter
import sys

root = next(path for path in (Path.cwd(), *Path.cwd().parents)
            if (path / "rust/crates/lobo_replay").is_dir())
sys.path.insert(0, str(root / "notebooks"))
from adapter_examples import wait_for_book
import pandas as pd
from IPython.display import display, HTML
from lobo.server import server_context

## Define the protocol

These declarations are the adapter definition. Edit the layouts, conditions or mappings here to change how your source drives the books.

In [2]:
from lobo.replay.adapters import CustomAdapter, Protocol
from lobo.replay.adapters import expressions as le
from lobo.replay.adapters import models as lm


def protocol() -> Protocol:
    symbol = le.Variable("symbol")
    channel = le.Variable("channel")
    flags = 32768 | 65536 | 131072
    order_side = le.Choose(le.Field(2).gt(0), "buy", "sell")
    order_quantity = le.Field(2).absolute().decimal(8)
    timestamp = le.Variable("clock")
    snapshot = le.Field(0).lookup("snapshots").ne(True)
    checksum = lm.Checksum(
        le.Field(2),
        view="orders",
        depth=25,
        sides=("buy", "sell"),
        fields=("id", "signed_quantity"),
        separator=":",
        interleave=True,
        format="ecmascript",
        signed=True,
        priority="id",
    )
    return Protocol(
        lm.Json(
            lm.Message(
                le.Field("event").eq("conf"),
                le.When(
                    le.Field("status").eq("OK") & le.Field("flags").eq(flags),
                    lm.Subscribe(),
                    otherwise=(lm.Fail("Required feed flags were rejected"),),
                ),
            ),
            lm.Message(
                le.Field("event").eq("subscribed"),
                le.When(
                    le.Field("channel").eq("trades")
                    | (le.Field("channel").eq("book") & le.Field("prec").eq("R0")),
                    le.Remember(
                        "channels",
                        le.Field("chanId"),
                        {
                            "symbol": le.Field("symbol").strip_prefix("t"),
                            "book": le.Field("channel").eq("book"),
                        },
                    ),
                    otherwise=(lm.Fail("Unexpected subscription"),),
                ),
            ),
            lm.Message(
                le.Field("event").eq("error") | le.Field("event").eq("unsubscribed"),
                lm.Fail("Subscription failed or ended; reconnect required"),
            ),
            lm.Message(
                le.Field("event").eq("info"),
                le.When(
                    (le.Field("version").exists() & le.Field("version").ne(2))
                    | le.Field("platform", "status").eq(0)
                    | le.Field("code").map(
                        {20051: True, 20060: True, 20061: True}, default=False
                    ),
                    lm.Fail("Feed restart or unsupported version; reconnect required"),
                ),
            ),
            lm.Message(
                le.Field().is_array(),
                le.CheckSequence(le.Field(-2)),
                le.Let("clock", le.Field(-1).timestamp("ms")),
                le.Let("channel", le.Field(0).lookup("channels")),
                le.When(channel.exists(), otherwise=(lm.Fail("Unknown channel"),)),
                le.Let("symbol", channel.get("symbol")),
                le.When(
                    channel.get("book").eq(True),
                    le.When(
                        le.Field(1).eq("cs"),
                        lm.Book(
                            channel.get("symbol"),
                            checksum=checksum,
                            timestamp=timestamp,
                        ),
                    ),
                    le.When(
                        le.Field(1).is_array(),
                        lm.Book(
                            channel.get("symbol"),
                            le.When(
                                snapshot,
                                le.ForEach(
                                    le.Field(1),
                                    lm.Add(
                                        id=le.Field(0),
                                        side=order_side,
                                        price=le.Field(1).decimal(8),
                                        quantity=order_quantity,
                                    ),
                                    order_by=le.Field(0),
                                    unique_by=le.Field(0),
                                ),
                                otherwise=(
                                    le.ForEach(
                                        [le.Root(1)],
                                        lm.Upsert(
                                            id=le.Field(0),
                                            side=order_side,
                                            price=le.Choose(
                                                le.Field(1).eq(0),
                                                None,
                                                le.Field(1).decimal(8),
                                            ),
                                            quantity=order_quantity,
                                        ),
                                    ),
                                ),
                            ),
                            snapshot=snapshot,
                            timestamp=timestamp,
                            ready=False,
                        ),
                        le.Remember("snapshots", le.Field(0), True),
                    ),
                    otherwise=(
                        le.When(
                            le.Field(1).eq("te") | le.Field(1).eq("tu"),
                            lm.Book(
                                channel.get("symbol"),
                                le.ForEach(
                                    [le.Root(2)],
                                    lm.Trade(
                                        id=le.Field(0),
                                        price=le.Field(3).decimal(8),
                                        quantity=le.Field(2).absolute().decimal(8),
                                        maker_side=le.Choose(
                                            le.Field(2).gt(0), "sell", "buy"
                                        ),
                                        timestamp=le.Field(1).timestamp("ms"),
                                    ),
                                ),
                                timestamp=timestamp,
                                ready=False,
                            ),
                        ),
                        le.When(
                            le.Field(1).is_array(),
                            le.ForEach(le.Field(1), lm.TradeHistory(le.Field(0))),
                        ),
                    ),
                ),
            ),
        ),
        bootstrap=(
            lm.Bootstrap(
                "instruments",
                "https://api-pub.bitfinex.com/v2/conf/pub:list:pair:exchange",
                le.ForEach(
                    le.Field(0),
                    lm.Register(symbol=le.Field(), price_decimals=8, quantity_decimals=8),
                ),
            ),
        ),
        connect=(lm.Send({"event": "conf", "flags": flags}),),
        subscriptions=(
            lm.Send(
                {
                    "event": "subscribe",
                    "channel": "book",
                    "symbol": le.Concat("t", symbol),
                    "prec": "R0",
                    "freq": "F0",
                    "len": "250",
                }
            ),
            lm.Send(
                {
                    "event": "subscribe",
                    "channel": "trades",
                    "symbol": le.Concat("t", symbol),
                }
            ),
        ),
        keepalive=(lm.Send({"event": "ping"}),),
        symbols_per_connection=15,
        simulation_note="Estimated FIFO from visible orders and public trades; corrections reconcile for 250 ms.",
    )

## Construct, then consume input

`CustomAdapter(protocol(), source, ...)` validates and compiles the declaration with Cranelift during construction. There is no separate compile call. The source then supplies raw bytes to the compiled packet program. Changing a declaration creates a different program; identical definitions can reuse the process-local cache.

The timings below separate construction from `start()` → `wait()`. Construction includes declaration conversion and preparation; it is **not a compiler-only measurement**. Starting also includes worker setup and, when serving, preparation for publishing to observers. These tiny examples demonstrate behavior, not throughput.

JSON uses the compiled Rust parser to produce a value tree, followed by generated field extraction and control flow. It still allocates that tree. Definition validation is done at construction; incoming syntax, sequence and checksum handling remain part of consuming the feed.

In [3]:
recording = root / "rust/crates/lobo_adapters/tests/fixtures/bitfinex_r0.jsonl"
source = lm.Source.json_lines(recording, bootstrap=[("instruments", b'[["BTCUSD"]]')])
definition = protocol()
started = perf_counter()
fixture_adapter = CustomAdapter(
    definition, source, name="Bitfinex recording", symbol="BTCUSD", scope=["BTCUSD"],
    mode="live", level="l3",
)
construction_ms = (perf_counter() - started) * 1000

with fixture_adapter:
    started = perf_counter()
    fixture_adapter.start()
    fixture_adapter.wait()
    completion_ms = (perf_counter() - started) * 1000
    status = fixture_adapter.status()
    levels = pd.DataFrame(fixture_adapter.levels("BTCUSD"))
    assert status["messages"] == 294
    assert status["checksum_checks"] > 0 and status["checksum_failures"] == 0
    display(levels.head(12))
    queue = pd.DataFrame(fixture_adapter.queue("BTCUSD", "buy", 0, 2**64 - 1),
                         columns=["order_id", "price", "quantity", "timestamp_ns"])
    display(queue.head(10))
    fixture_adapter.simulate("buy", 1_000_000)
    display(pd.DataFrame(fixture_adapter.simulation_report()["executions"]))

print(f"Construction: {construction_ms:.3f} ms | Start to completion: {completion_ms:.3f} ms")
print(f"Consumed {status['bytes']:,} bytes / {status['messages']} messages")
print(f"Checksums: {status['checksum_checks']} passed, {status['checksum_failures']} failed")

,hidden,orders,price,quantity,side
0,0,1,7942500000000,22000,buy
1,0,1,7941900000000,2300000,buy
2,0,2,7941800000000,404723,buy
3,0,1,7941700000000,22000,buy
4,0,1,7941600000000,22000,buy
5,0,1,7941400000000,24450,buy
6,0,1,7941300000000,22000,buy
7,0,2,7941100000000,2528542,buy
8,0,2,7941000000000,7282303,buy
9,0,1,7940900000000,22000,buy


,order_id,price,quantity,timestamp_ns
0,243623296906,7942500000000,22000,1788829786770000000
1,243628348659,7941900000000,2300000,1788829784599000000
2,243624408837,7941800000000,22000,1788829784599000000
3,243628208957,7941800000000,382723,1788829784599000000
4,243617778172,7941700000000,22000,1788829787102000000
5,243627785657,7941600000000,22000,1788829784599000000
6,243628122287,7941400000000,24450,1788829784599000000
7,243617700294,7941300000000,22000,1788829784599000000
8,243602358189,7941100000000,10000,1788829789170000000
9,243628351111,7941100000000,2518542,1788829789917000000


,price,quantity,sequence,simulated,timestamp_ns
0,7944000000000,1000000,1,True,1788829789917000000


Construction: 4.039 ms | Start to completion: 3.577 ms
Consumed 32,537 bytes / 294 messages
Checksums: 2 passed, 0 failed


## Connect the source and serve its books

The same declaration drives the public WebSocket. The scope fixes which books the session runs. This section requires internet access; no credentials are needed.

In [4]:
symbol = 'BTCUSD'
definition = protocol()
started = perf_counter()
adapter = CustomAdapter(
    definition, lm.Source.websocket("wss://api-pub.bitfinex.com/ws/2"),
    name='Bitfinex R0', symbol=symbol, scope=[symbol], mode='live', level='l3',
    timezone='UTC',
)
print(f"Construction (may reuse compiled code): {(perf_counter() - started) * 1000:.3f} ms")

Construction (may reuse compiled code): 0.386 ms


Open the link to see the charts and use the simulation controls. The server stays running until the cleanup cell.

Run the cells individually to keep the terminal open while you explore. Run All reaches cleanup and closes it.

In [5]:
terminal = server_context(adapters=[adapter], port=0)
display(HTML(f'<a href="{terminal.url}" target="_blank">Open order-book terminal</a>'))

In [6]:
levels = wait_for_book(adapter, symbol)
display(pd.DataFrame(levels).head(12))
adapter.status()

,hidden,orders,price,quantity,side
0,0,1,7897200000000,3165656,buy
1,0,1,7897100000000,2300000,buy
2,0,2,7897000000000,6016717,buy
3,0,1,7896800000000,22000,buy
4,0,2,7896600000000,12683548,buy
5,0,1,7896300000000,5046660,buy
6,0,1,7896200000000,539654,buy
7,0,2,7896000000000,5496581,buy
8,0,2,7895900000000,13193885,buy
9,0,1,7895800000000,22000,buy


{'books': 1,
 'bytes': 29174,
 'checksum_checks': 1,
 'checksum_failures': 0,
 'clock_ns': 1788932745289809584,
 'complete': False,
 'messages': 245,
 'symbol': 'BTCUSD',
 'synchronized': True}

Preview a market order and inspect its execution report.

In [7]:
adapter.simulate("buy", 1000000)
adapter.simulation_report()

{'alternate_timeline': False,
 'average_price': 7898500000000.0,
 'complete': True,
 'executions': [{'price': 7898500000000,
   'quantity': 1000000,
   'sequence': 1,
   'simulated': True,
   'timestamp_ns': 1788932745299876584}],
 'filled': 1000000,
 'ignored': 0,
 'order_id': '054a435d-60d2-44e7-9b28-4b3b1e854afd',
 'remaining': 0,
 'requested': 1000000,
 'simulated': True,
 'stopped': True,
 'symbol': 'BTCUSD'}

Run this cell when finished exploring the terminal.

In [8]:
terminal.close()

## Recorded performance results

The September 9, 2026 paired benchmark for **294 recorded frames** measured **482.3 µs** for the existing adapter and **775.3 µs** for the compiled custom adapter, a **1.61×** paired ratio. Streaming parity has not been reached. These are saved benchmark results, not timings from this notebook. The benchmark excludes construction and compilation; the demonstration above includes worker startup.

See [validation and reproduction commands](../python/examples/VALIDATION.md) for all four feeds and the timing boundaries.